In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
 
train_df = pd.read_csv('train_preprocessed.csv')
test_df = pd.read_csv('test_preprocessed.csv')
 
X_train = train_df.drop('RainTomorrow', axis=1)
y_train = train_df['RainTomorrow']
X_test = test_df.drop('RainTomorrow', axis=1)
y_test = test_df['RainTomorrow']

X_tune = X_train.sample(n=30_000, random_state=42)
y_tune = y_train.loc[X_tune.index]
 
lr_pipeline = Pipeline([
    ('lr', LogisticRegression(
        solver='saga',
        max_iter=500,
        random_state=42
    ))
])
 
param_grid = {
    'lr__l1_ratio': [0.0, 0.5, 1.0],  
    'lr__C': [0.01, 0.1, 0.5, 1.0],
}
 
search = GridSearchCV(
    lr_pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)
search.fit(X_tune, y_tune)
 
results = pd.DataFrame(search.cv_results_)
cols = ['param_lr__l1_ratio', 'param_lr__C', 'mean_test_score', 'rank_test_score']
report_df = results[cols].sort_values(by='mean_test_score', ascending=False)
report_df.columns = ['l1_ratio', 'C', 'avg_F1-Score', 'rank']
print(report_df.to_string(index=False))
 
best_lr = search.best_estimator_
 
best_lr.fit(X_train, y_train)
 
y_pred = best_lr.predict(X_test)
y_pred_proba = best_lr.predict_proba(X_test)[:, 1]
 
print(classification_report(y_test, y_pred, target_names=['No Rain (0)', 'Rain (1)']))
 
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"ROC AUC Score: {auc_score:.4f}")
 
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
 
feature_names = X_train.columns.tolist()
coefs = best_lr.named_steps['lr'].coef_[0]
best_l1_ratio = search.best_params_['lr__l1_ratio']
penalty_label = {0.0: 'L2 (Ridge)', 1.0: 'L1 (Lasso)'}.get(best_l1_ratio, f'ElasticNet (l1_ratio={best_l1_ratio})')
 
importance_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefs,
    'abs_coefficient': np.abs(coefs)
}).sort_values('abs_coefficient', ascending=False)
 
print(f"\nTop 15 most influential features (best penalty: {penalty_label}):")
print(importance_df.head(15).to_string(index=False))
 
if best_l1_ratio > 0:
    n_zero = (coefs == 0).sum()
    print(f"\nL1 sparsity: {n_zero} / {len(feature_names)} features zeroed out "
          f"({n_zero / len(feature_names) * 100:.1f}% eliminated)")
 

/Users/yangseongwon/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/yangseongwon/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/yangseongwon/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/yangseongwon/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/yangseongwon/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/yangseongwon/minic

 l1_ratio    C  avg_F1-Score  rank
      1.0 1.00      0.787843     1
      0.5 0.50      0.787803     2
      1.0 0.50      0.787671     3
      0.5 1.00      0.787545     4
      0.0 0.10      0.787489     5
      0.0 1.00      0.787397     6
      0.0 0.50      0.787258     7
      0.5 0.10      0.787065     8
      1.0 0.10      0.786624     9
      0.0 0.01      0.782816    10
      0.5 0.01      0.775907    11
      1.0 0.01      0.772635    12
              precision    recall  f1-score   support

 No Rain (0)       0.92      0.80      0.86     22064
    Rain (1)       0.53      0.77      0.63      6375

    accuracy                           0.79     28439
   macro avg       0.73      0.79      0.74     28439
weighted avg       0.84      0.79      0.81     28439

ROC AUC Score: 0.8714

Confusion Matrix:
[[17663  4401]
 [ 1444  4931]]

Top 15 most influential features (best penalty: L1 (Lasso)):
                  feature  coefficient  abs_coefficient
     Location_MountGinini   